## 1. Paths and imports


# TabPFN FP follow-up (VLST): ratio sweep + dual evaluation matrices

1. Loads **VLST** + union **`false_positives_union_test.csv`**; aligns features. **By default drops** `NO.` and `Name` (`VLST_DROP_ROW_ID_COLS` — set to `0` / `false` / `no` to keep). Those columns often yield **AUC ≈ 1** (row memorization).
2. Cohort per ratio **r**: all positives + mined FPs first + TNs until `92×r` negatives. Ratios = **1**, then **5, 10, …**, **plus `max_r = ⌊max_neg_total/92⌋`** if not already listed (so the last grid step uses ~all negatives, e.g. **44** not only **40**), then optional **full_pool** if it differs.
3. **Stratified 60/20/20** → **TabPFN** fits **train 60%** only. **split_a** = cohort **holdout** (20%, has positives). **split_b** = full eval pool minus rows in the **fit** fold.
4. **Three thresholds** on both splits (printed + CSV columns `a_t05_*`, `a_tval_*`, `a_thold_*`, same for **b_**):
   - **0.5** — each ratio step prints **two** confusion matrices at **t=0.5** (split_a then split_b) before the val/holdout threshold blocks
   - **t_val** = argmax **F0.5 on val** (recommended for reporting)
   - **t_hold** = argmax **F0.5 on cohort holdout** (oracle — optimistic; do not use for model selection)
5. **ROC-AUC / PR-AUC** are computed from **predicted probabilities on that eval split** (split_a or split_b). They are **not** a single global score from train/val. **split_a** and **split_b** each get their own PR-AUC; within one split, `*_t05_pr_auc`, `*_tval_pr_auc`, and `*_thold_pr_auc` are identical (AUC ignores the classification threshold).
6. **`fp_followup_ratio_sweep.csv`**, §4 shows a **scrollable** HTML table when IPython is available.

**Backend:** `tabpfn-client` (cloud).

**Env:** `VLST_FULL_DATA_PATH`, `VLST_FP_OUTPUT_DIR`, `TABPFN_TOKEN`, `VLST_TABPFN_FOLLOWUP_SEED`, `VLST_DROP_ROW_ID_COLS`, `TABPFN_N_ESTIMATORS`.



In [1]:
import os
import warnings

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    fbeta_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore", category=UserWarning)

RANDOM_STATE = int(os.environ.get("VLST_TABPFN_FOLLOWUP_SEED", "42"))

FULL_DATA_PATH = os.environ.get(
    "VLST_FULL_DATA_PATH",
    "/kaggle/input/datasets/amirmahdidaraei/vlst-data/VLST.csv",
)


def pick_output_dir_modeling_fp() -> str:
    out = os.environ.get("VLST_FP_FOLLOWUP_OUT_DIR")
    if out:
        return os.path.expanduser(out.rstrip(os.sep))
    if os.path.isdir("/kaggle/working"):
        return "/kaggle/working/vlst_fp_followup_output"
    return os.path.normpath(os.path.join("..", "..", "data", "result", "modeling_fp"))


OUTPUT_DIR = pick_output_dir_modeling_fp()
os.makedirs(OUTPUT_DIR, exist_ok=True)


def pick_fp_mining_output_dir() -> str:
    out = os.environ.get("VLST_FP_OUTPUT_DIR") or os.environ.get("VLST_FP_MINING_OUT")
    if not out:
        kaggle_fp_out = "/kaggle/input/datasets/amirmahdidaraei/fp-output/vlst_fp_mining_output"
        if os.path.isdir(kaggle_fp_out):
            out = kaggle_fp_out
    if out:
        out = os.path.expanduser(out.rstrip(os.sep))
    elif os.path.isdir("/kaggle/working"):
        out = "/kaggle/working/vlst_fp_mining_output"
    else:
        repo_modeling_fp = os.path.normpath(
            os.path.join("..", "..", "data", "result", "modeling_fp")
        )
        if os.path.isfile(os.path.join(repo_modeling_fp, "false_positives_union_test.csv")):
            out = repo_modeling_fp
        else:
            out = os.path.normpath(
                os.path.join("..", "..", "data", "result", "modeling_advanced", "fp_mining")
            )
    tag = os.environ.get("VLST_FP_RUN_TAG", "").strip()
    if tag:
        out = os.path.join(out, tag)
    return out


FP_MINING_OUT = pick_fp_mining_output_dir()
UNION_CSV = os.path.join(FP_MINING_OUT, "false_positives_union_test.csv")

print("FULL_DATA_PATH:", FULL_DATA_PATH)
print("Follow-up artifacts dir (OUTPUT_DIR):", OUTPUT_DIR)
print("FP mining outputs (union CSV):", FP_MINING_OUT)
print("Union CSV:", UNION_CSV)


FULL_DATA_PATH: /kaggle/input/datasets/amirmahdidaraei/vlst-data/VLST.csv
Follow-up artifacts dir (OUTPUT_DIR): /kaggle/working/vlst_fp_followup_output
FP mining outputs (union CSV): /kaggle/input/datasets/amirmahdidaraei/fp-output/vlst_fp_mining_output
Union CSV: /kaggle/input/datasets/amirmahdidaraei/fp-output/vlst_fp_mining_output/false_positives_union_test.csv


In [2]:
!pip install -U tabpfn-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 788.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.8/240.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests=

In [3]:
from tabpfn_client import set_access_token

_token = os.environ.get("TABPFN_TOKEN", "").strip()
if not _token:
    try:
        from kaggle_secrets import UserSecretsClient

        _token = str(UserSecretsClient().get_secret("TABPFN_TOKEN")).strip()
        os.environ["TABPFN_TOKEN"] = _token
    except Exception:
        pass

if _token:
    set_access_token(_token)
    print("TabPFN client: access token configured (value hidden).")
else:
    print(
        "Set TABPFN_TOKEN (Kaggle Secret, shell export, or os.environ) before §3. "
        "Get a key at https://ux.priorlabs.ai/account"
    )


TabPFN client: access token configured (value hidden).


## 2. Load data, align features, train/test pools, ratio schedule


In [4]:
if not os.path.isfile(FULL_DATA_PATH):
    raise FileNotFoundError(
        f"Missing full dataset at {FULL_DATA_PATH}. Set VLST_FULL_DATA_PATH before running."
    )
if not os.path.isfile(UNION_CSV):
    raise FileNotFoundError(
        f"Missing {UNION_CSV}. Run fp mining export first or set VLST_FP_OUTPUT_DIR / VLST_FP_RUN_TAG."
    )

df_full = pd.read_csv(FULL_DATA_PATH, low_memory=False)
df_union = pd.read_csv(UNION_CSV, low_memory=False)
if "test_row_id" not in df_union.columns:
    raise ValueError("Union CSV must contain test_row_id.")


def _norm_col(s: str) -> str:
    return "".join(ch.lower() for ch in str(s).strip() if ch.isalnum())


col_map = {_norm_col(c): c for c in df_full.columns}

forced_target = os.environ.get("VLST_FULLDATA_TARGET_COL", "").strip()
if forced_target:
    fk = _norm_col(forced_target)
    if fk not in col_map:
        raise ValueError(
            f"VLST_FULLDATA_TARGET_COL={forced_target!r} not found. "
            f"Available columns include: {list(df_full.columns)[:12]} ..."
        )
    target_col = col_map[fk]
else:
    preferred = [
        "Stent thrombosis",
        "stent_thrombosis",
        "StentThrombosis",
        "target",
        "label",
        "y",
        "class",
        "Outcome",
        "VLST",
    ]
    target_col = None
    for c in preferred:
        k = _norm_col(c)
        if k in col_map:
            cand = col_map[k]
            vals = set(
                pd.to_numeric(df_full[cand], errors="coerce")
                .dropna()
                .astype(int)
                .unique()
                .tolist()
            )
            if vals.issubset({0, 1}) and len(vals) >= 1:
                target_col = cand
                break
    if target_col is None:
        raise ValueError(
            "Could not confidently infer target column. "
            "Set VLST_FULLDATA_TARGET_COL='Stent thrombosis' (or your label column name)."
        )

print("Using target column from full data:", target_col)

y_full = pd.to_numeric(df_full[target_col], errors="coerce").fillna(0).astype(int).to_numpy()
if not set(np.unique(y_full)).issubset({0, 1}):
    raise ValueError("Target column must be binary (0/1).")

X_full_df = df_full.drop(columns=[target_col]).copy()

_drop_ids = os.environ.get("VLST_DROP_ROW_ID_COLS", "1").strip().lower() not in (
    "0",
    "false",
    "no",
)
if _drop_ids:
    _id_cols = [c for c in ("NO.", "Name") if c in X_full_df.columns]
    if _id_cols:
        X_full_df = X_full_df.drop(columns=_id_cols)
        print("Dropped row-identifier columns from features:", _id_cols)


def _norm_name(s: str) -> str:
    return "".join(ch.lower() for ch in str(s) if ch.isalnum())


_drop_norm = {
    _norm_name("Time since stent implantation"),
    _norm_name("time_since_implantation"),
    _norm_name("time since implantation"),
}
_drop_cols = [c for c in X_full_df.columns if _norm_name(c) in _drop_norm]
if _drop_cols:
    X_full_df = X_full_df.drop(columns=_drop_cols)
    print("Dropped leakage-style feature(s):", _drop_cols)

for c in X_full_df.columns:
    X_full_df[c] = pd.to_numeric(X_full_df[c], errors="coerce")

meta_cols = {
    "test_row_id",
    "y_true",
    "p_winner",
    "p_tabpfn",
    "fp_winner",
    "fp_tabpfn_legacy_t",
    "fp_tabpfn_any_grid_t",
    "source",
    "threshold",
}
shared_features = [c for c in X_full_df.columns if c not in meta_cols]
if not shared_features:
    raise ValueError("No feature columns in full dataset after exclusions.")

_union_feat = [c for c in df_union.columns if c not in meta_cols]
union_only_cols = [c for c in _union_feat if c not in X_full_df.columns]
if union_only_cols:
    print(
        "Note: union CSV has",
        len(union_only_cols),
        "feature column(s) not in full data (omitted from training schema); examples:",
        union_only_cols[:12],
    )

X_full_use = X_full_df[shared_features].copy()
X_union_use = df_union.reindex(columns=shared_features)
for c in shared_features:
    X_union_use[c] = pd.to_numeric(X_union_use[c], errors="coerce")

med = X_full_use.median(numeric_only=True)
X_full_use = X_full_use.fillna(med)
X_union_use = X_union_use.fillna(med)

_union_ids = pd.to_numeric(df_union["test_row_id"], errors="coerce")
_rid = np.where(np.isfinite(_union_ids.to_numpy(dtype=float)), _union_ids.to_numpy(dtype=float), -1.0).astype(np.int64)
_valid_overlay = (_rid >= 0) & (_rid < len(X_full_df))
if int(_valid_overlay.sum()) > 0:
    _vi = np.flatnonzero(_valid_overlay)
    X_union_use.iloc[_vi, :] = X_full_use.iloc[_rid[_vi]].to_numpy(dtype=np.float32, copy=False)
    print(
        "Union FP rows overlaid from full VLST by test_row_id:",
        int(_valid_overlay.sum()),
        "/",
        len(df_union),
    )
else:
    print(
        "Warning: no union test_row_id in [0, n_rows(VLST)); FP rows stay union CSV + median only."
    )

assert (
    X_full_use.shape[1] == len(shared_features) == X_union_use.shape[1]
), (X_full_use.shape[1], len(shared_features), X_union_use.shape[1])

rng = np.random.RandomState(RANDOM_STATE)
pos_idx = np.flatnonzero(y_full == 1)
tn_pool_idx = np.flatnonzero(y_full == 0)
if pos_idx.size == 0:
    raise ValueError("No positives in full dataset.")

union_full_row_ids = set(int(x) for x in df_union["test_row_id"].astype(int).tolist())

# Eligible true negatives for TN fill / test reserve: label 0 and not a union row id
tn_elig = np.array(
    sorted(i for i in tn_pool_idx.tolist() if int(i) not in union_full_row_ids),
    dtype=int,
)
if tn_elig.size < 32:
    tn_elig = np.array(sorted(tn_pool_idx.tolist()), dtype=int)

# Reserve a slice of eligible TNs for the "left-out" pool (never in cohort TN fill)
TEST_NEG_FRAC = float(os.environ.get("VLST_FP_FOLLOWUP_TEST_NEG_FRAC", "0.2"))
if not (0.0 < TEST_NEG_FRAC < 0.5):
    raise ValueError("VLST_FP_FOLLOWUP_TEST_NEG_FRAC must be in (0, 0.5).")
ix_tn = np.arange(len(tn_elig), dtype=int)
tn_train_ix, tn_test_ix = train_test_split(
    ix_tn,
    test_size=TEST_NEG_FRAC,
    random_state=RANDOM_STATE,
    shuffle=True,
)
TestFixed_neg = tn_elig[tn_test_ix]  # extra negatives for split_b only
tn_for_train_pool = tn_elig[tn_train_ix]

# Deterministic shuffle order for consuming extra negatives across ratio steps
perm_tn = rng.permutation(len(tn_for_train_pool))
tn_ordered = tn_for_train_pool[perm_tn]

fp_mat_all = X_union_use.to_numpy(dtype=np.float32)
n_fp = int(fp_mat_all.shape[0])
n_pos = int(pos_idx.size)

X_pos_all = X_full_use.iloc[pos_idx].to_numpy(dtype=np.float32)
y_pos_vec = np.ones(n_pos, dtype=int)

max_neg_total = n_fp + int(tn_ordered.size)
max_r = int(max_neg_total // n_pos) if n_pos else 0
ratio_candidates = [1] + list(range(5, max(max_r, 5) + 1, 5))
if max_r >= 1 and max_r not in ratio_candidates:
    ratio_candidates.append(max_r)
ratio_candidates = sorted(set(int(x) for x in ratio_candidates))
RATIOS = []
for r in ratio_candidates:
    need = n_pos * int(r)
    if need <= max_neg_total:
        RATIOS.append(int(r))

if not RATIOS:
    raise RuntimeError("No valid ratio steps (check TN pool and n_fp).")

# Full-pool negative count (all FPs + all train-pool TNs)
n_neg_full = n_fp + int(tn_ordered.size)

split_summary = {
    "n_pos": n_pos,
    "n_fp_union": n_fp,
    "n_tn_train_pool": int(tn_ordered.size),
    "n_TestFixed_neg": int(TestFixed_neg.size),
    "max_neg_total": max_neg_total,
    "max_r_neg_ratio": int(max_r),
    "ratios": RATIOS,
    "n_neg_full_pool": int(n_neg_full),
    "TEST_NEG_FRAC": TEST_NEG_FRAC,
}
print("Split / ratio summary:", split_summary)
print("Features (shared_features):", len(shared_features))


def stratified_602020(X_mat: np.ndarray, y_vec: np.ndarray, random_state: int):
    """60/20/20 stratified split; returns folds and index arrays."""
    idx = np.arange(len(y_vec), dtype=int)
    i_tr, i_temp = train_test_split(
        idx,
        test_size=0.4,
        random_state=random_state,
        shuffle=True,
        stratify=y_vec,
    )
    i_va, i_ho = train_test_split(
        i_temp,
        test_size=0.5,
        random_state=random_state,
        shuffle=True,
        stratify=y_vec[i_temp],
    )
    return (
        X_mat[i_tr],
        y_vec[i_tr],
        X_mat[i_va],
        y_vec[i_va],
        X_mat[i_ho],
        y_vec[i_ho],
        i_tr,
        i_va,
        i_ho,
    )


def build_train_arrays(n_neg_target: int, rng_step: np.random.RandomState):
    """Cohort: all positives + mined FPs first + TN fill. Returns full-table row id per row."""
    need = int(min(n_neg_target, n_fp + tn_ordered.size))
    n_fp_take = min(n_fp, need)
    need_after_fp = need - n_fp_take
    X_fp_part = fp_mat_all[:n_fp_take] if n_fp_take > 0 else np.empty((0, fp_mat_all.shape[1]), dtype=np.float32)
    tn_take = int(min(need_after_fp, tn_ordered.size))
    if tn_take > 0:
        tn_idx_used = tn_ordered[:tn_take]
        X_tn_part = X_full_use.iloc[tn_idx_used].to_numpy(dtype=np.float32)
    else:
        tn_idx_used = np.array([], dtype=int)
        X_tn_part = np.empty((0, fp_mat_all.shape[1]), dtype=np.float32)

    full_ix_list = (
        [int(i) for i in pos_idx.tolist()]
        + [int(df_union.iloc[j]["test_row_id"]) for j in range(n_fp_take)]
        + [int(i) for i in tn_idx_used.tolist()]
    )
    X_all = np.vstack([X_pos_all, X_fp_part, X_tn_part]).astype(np.float32)
    y_all = np.concatenate(
        [
            y_pos_vec,
            np.zeros(n_fp_take, dtype=int),
            np.zeros(tn_take, dtype=int),
        ]
    )
    full_ix = np.asarray(full_ix_list, dtype=np.int64)
    perm = rng_step.permutation(len(y_all))
    X_all, y_all, full_ix = X_all[perm], y_all[perm], full_ix[perm]
    return X_all, y_all, full_ix, n_fp_take, tn_idx_used


def pool_full_idx_set():
    return (
        set(int(x) for x in pos_idx.tolist())
        | set(int(x) for x in tn_elig.tolist())
        | set(int(x) for x in union_full_row_ids)
    )


def Xy_for_full_indices(indices: np.ndarray):
    # Features/labels for full-table row indices.
    indices = np.asarray(indices, dtype=int)
    Xb = X_full_use.iloc[indices].to_numpy(dtype=np.float32)
    yb = y_full[indices].astype(int)
    return Xb, yb


def matrix_b_indices(train_fit_full_idx: set):
    """Full-table rows in eval pool but not in TabPFN .fit() train fold."""
    pool = pool_full_idx_set()
    return np.asarray(sorted(pool - set(train_fit_full_idx)), dtype=int)





Using target column from full data: Stent thrombosis
Dropped row-identifier columns from features: ['NO.', 'Name']
Dropped leakage-style feature(s): ['Time since stent implantation']
Note: union CSV has 93 feature column(s) not in full data (omitted from training schema); examples: ['Stent type-SES_Endeavor', 'Stent type-SES_Ex', 'Stent type-SES_Ex/Pa', 'Stent type-SES_Excecl', 'Stent type-SES_Excel', 'Stent type-SES_Excel/Firebird', 'Stent type-SES_Excel/Partner', 'Stent type-SES_Excel:RCA', 'Stent type-SES_Firebid', 'Stent type-SES_Firebird', 'Stent type-SES_LAD:Excel', 'Stent type-SES_LAD:Firebird']
Union FP rows overlaid from full VLST by test_row_id: 44 / 44
Split / ratio summary: {'n_pos': 92, 'n_fp_union': 44, 'n_tn_train_pool': 4039, 'n_TestFixed_neg': 1010, 'max_neg_total': 4083, 'max_r_neg_ratio': 44, 'ratios': [1, 5, 10, 15, 20, 25, 30, 35, 40, 44], 'n_neg_full_pool': 4083, 'TEST_NEG_FRAC': 0.2}
Features (shared_features): 81


## 3. Ratio sweep — TabPFN, F0.5 threshold, dual confusion matrices + CSV


In [5]:
from tabpfn_client import TabPFNClassifier, set_access_token

_token = os.environ.get("TABPFN_TOKEN", "").strip()
if not _token:
    raise RuntimeError(
        "TABPFN_TOKEN missing. Run the token cell above (Kaggle Secret or env var). "
        "API key: https://ux.priorlabs.ai/account"
    )
set_access_token(_token)

TABPFN_N_ESTIMATORS = int(os.environ.get("TABPFN_N_ESTIMATORS", "8"))


def make_tabpfn_classifier(random_state: int) -> TabPFNClassifier:
    """tabpfn-client API (cloud); no local `device` argument."""
    return TabPFNClassifier(
        random_state=random_state,
        n_estimators=TABPFN_N_ESTIMATORS,
        ignore_pretraining_limits=True,
        balance_probabilities=True,
    )

t_grid = np.arange(0.01, 1.0, 0.01)


def best_threshold(y_true, p, grid, metric_fn):
    best_t, best_v = 0.5, -1.0
    for t in grid:
        y_hat = (p >= t).astype(int)
        v = float(metric_fn(y_true, y_hat))
        if v > best_v:
            best_v, best_t = v, float(t)
    return best_t, best_v


def f05m(y, yhat):
    return fbeta_score(y, yhat, beta=0.5, zero_division=0)


def metric_bundle(y_true, y_hat, p):
    out = {
        "precision": float(precision_score(y_true, y_hat, zero_division=0)),
        "recall": float(recall_score(y_true, y_hat, zero_division=0)),
        "f1": float(f1_score(y_true, y_hat, zero_division=0)),
        "f05": float(fbeta_score(y_true, y_hat, beta=0.5, zero_division=0)),
        "f2": float(fbeta_score(y_true, y_hat, beta=2.0, zero_division=0)),
        "accuracy": float(accuracy_score(y_true, y_hat)),
    }
    try:
        out["roc_auc"] = float(roc_auc_score(y_true, p))
    except ValueError:
        out["roc_auc"] = float("nan")
    try:
        out["pr_auc"] = float(average_precision_score(y_true, p))
    except ValueError:
        out["pr_auc"] = float("nan")
    return out


def print_confusion_matrix_text(cm):
    cm = np.asarray(cm, dtype=int)
    print("Confusion matrix (rows=true, cols=pred):")
    if cm.shape == (2, 2):
        print("                pred:0 (neg)   pred:1 (pos)")
        print(f"    true:0 (neg)   {cm[0, 0]:>10}   {cm[0, 1]:>10}   TN, FP")
        print(f"    true:1 (pos)   {cm[1, 0]:>10}   {cm[1, 1]:>10}   FN, TP")
    else:
        for row in cm:
            print("   ", " ".join(f"{v:>8}" for v in row))


def append_eval_row(d, prefix, y_true, p, t_use, label):
    y_hat = (p >= float(t_use)).astype(int)
    mb = metric_bundle(y_true, y_hat, p)
    for k, v in mb.items():
        d[f"{prefix}_{label}_{k}"] = float(v)
    cm = confusion_matrix(y_true, y_hat, labels=[0, 1])
    d[f"{prefix}_{label}_cm00"] = int(cm[0, 0])
    d[f"{prefix}_{label}_cm01"] = int(cm[0, 1])
    d[f"{prefix}_{label}_cm10"] = int(cm[1, 0])
    d[f"{prefix}_{label}_cm11"] = int(cm[1, 1])


def print_confusion_at_threshold(title, y_true, p, t):
    """Confusion matrix + threshold-dependent metrics at fixed t (ROC/PR-AUC from probabilities)."""
    t = float(t)
    y_hat = (p >= t).astype(int)
    mb = metric_bundle(y_true, y_hat, p)
    print(f"\n=== {title} @ threshold {t:.3f} ===")
    print(
        f"  precision={mb['precision']:.4f} recall={mb['recall']:.4f} "
        f"F0.5={mb['f05']:.4f} acc={mb['accuracy']:.4f} | "
        f"ROC-AUC={mb['roc_auc']:.4f} PR-AUC={mb['pr_auc']:.4f} (prob-based, same for any t on this split)"
    )
    print_confusion_matrix_text(confusion_matrix(y_true, y_hat, labels=[0, 1]))


def print_both_confusion_matrices_at_05(step_label, y_a, p_a, y_b, p_b):
    print("\n" + "-" * 72)
    print(f"{step_label} — confusion matrices @ threshold 0.5")
    print_confusion_at_threshold(
        f"{step_label} | matrix 1 split_a (cohort holdout, n={len(y_a)} pos={int(y_a.sum())})",
        y_a,
        p_a,
        0.5,
    )
    print_confusion_at_threshold(
        f"{step_label} | matrix 2 split_b (pool minus train-fit, n={len(y_b)} pos={int(y_b.sum())})",
        y_b,
        p_b,
        0.5,
    )


def print_three_thresholds(title, y, p, t_val, t_hold):
    print("\n===", title, "===")
    for t, lab in [
        (0.5, "t=0.5"),
        (t_val, "t=val_best_F0.5"),
        (t_hold, "t=holdout_oracle_F0.5 (optimistic)"),
    ]:
        yh = (p >= float(t)).astype(int)
        mb = metric_bundle(y, yh, p)
        print(
            f"  {lab:42s} t={float(t):5.3f} | ROC-AUC={mb['roc_auc']:.4f} PR-AUC={mb['pr_auc']:.4f} "
            f"F0.5={mb['f05']:.4f} acc={mb['accuracy']:.4f}"
        )
        print_confusion_matrix_text(confusion_matrix(y, yh, labels=[0, 1]))


sweep_rows = []
step_i = 0

for r in RATIOS:
    n_neg_target = n_pos * int(r)
    rs = np.random.RandomState(RANDOM_STATE + step_i * 9973)
    X_all, y_all, full_ix, n_fp_take, tn_used = build_train_arrays(n_neg_target, rs)
    X_tr, y_tr, X_va, y_va, X_ho, y_ho, i_tr, i_va, i_ho = stratified_602020(
        X_all, y_all, RANDOM_STATE + step_i
    )
    train_fit_ix = set(int(x) for x in full_ix[i_tr].tolist())

    clf = make_tabpfn_classifier(RANDOM_STATE + step_i)
    clf.fit(X_tr, y_tr)
    p_va = clf.predict_proba(X_va)[:, 1]
    t_val, v_f05_va = best_threshold(y_va, p_va, t_grid, f05m)

    p_a = clf.predict_proba(X_ho)[:, 1]
    t_hold, v_f05_hold = best_threshold(y_ho, p_a, t_grid, f05m)

    print("\n" + "=" * 72)
    print(
        f"ratio={r} | cohort holdout n={len(y_ho)} pos={int(y_ho.sum())} | "
        f"t_val={t_val:.4f} (val F0.5={v_f05_va:.4f}) | "
        f"t_hold_oracle={t_hold:.4f} (hold F0.5={v_f05_hold:.4f})"
    )

    b_ix = matrix_b_indices(train_fit_ix)
    X_b, y_b = Xy_for_full_indices(b_ix)
    p_b = clf.predict_proba(X_b)[:, 1]
    print_both_confusion_matrices_at_05(f"ratio={r}", y_ho, p_a, y_b, p_b)

    print_three_thresholds(f"ratio={r} split_a (cohort holdout)", y_ho, p_a, t_val, t_hold)
    print_three_thresholds(
        f"ratio={r} split_b (pool minus train-fit, n={len(y_b)} pos={int(y_b.sum())})",
        y_b,
        p_b,
        t_val,
        t_hold,
    )

    row = {
        "step_kind": "ratio",
        "ratio_requested": int(r),
        "n_neg_target": int(n_neg_target),
        "n_fp_in_constructed": int(n_fp_take),
        "n_tn_in_constructed": int(len(tn_used)),
        "n_fit_train": int(X_tr.shape[0]),
        "n_fit_val": int(X_va.shape[0]),
        "t_val_best_f05": float(t_val),
        "val_f05_at_t_val": float(v_f05_va),
        "t_hold_oracle_best_f05": float(t_hold),
        "hold_f05_at_t_hold": float(v_f05_hold),
        "split_a_n": int(len(y_ho)),
        "split_a_pos": int(y_ho.sum()),
        "split_b_n": int(len(y_b)),
        "split_b_pos": int(y_b.sum()),
    }
    for prefix, yv, pv in (("a", y_ho, p_a), ("b", y_b, p_b)):
        append_eval_row(row, prefix, yv, pv, 0.5, "t05")
        append_eval_row(row, prefix, yv, pv, t_val, "tval")
        append_eval_row(row, prefix, yv, pv, t_hold, "thold")
    sweep_rows.append(row)
    step_i += 1

rs = np.random.RandomState(RANDOM_STATE + step_i * 9973)
X_all_f, y_all_f, full_ix_f, n_fp_f, tn_used_f = build_train_arrays(n_neg_full, rs)
last_same = False
if sweep_rows:
    last = sweep_rows[-1]
    last_same = (
        int(last["n_fp_in_constructed"]) == int(n_fp_f)
        and int(last["n_tn_in_constructed"]) == int(len(tn_used_f))
    )
if not last_same:
    X_tr, y_tr, X_va, y_va, X_ho_f, y_ho_f, i_tr_f, _, _ = stratified_602020(
        X_all_f, y_all_f, RANDOM_STATE + step_i
    )
    train_fit_ix_f = set(int(x) for x in full_ix_f[i_tr_f].tolist())

    clf = make_tabpfn_classifier(RANDOM_STATE + step_i)
    clf.fit(X_tr, y_tr)
    p_va = clf.predict_proba(X_va)[:, 1]
    t_val, v_f05_va = best_threshold(y_va, p_va, t_grid, f05m)

    p_a = clf.predict_proba(X_ho_f)[:, 1]
    t_hold, v_f05_hold = best_threshold(y_ho_f, p_a, t_grid, f05m)

    print("\n" + "=" * 72)
    print(
        f"full_pool | holdout n={len(y_ho_f)} pos={int(y_ho_f.sum())} | "
        f"t_val={t_val:.4f} | t_hold_oracle={t_hold:.4f}"
    )

    b_ix = matrix_b_indices(train_fit_ix_f)
    X_b, y_b = Xy_for_full_indices(b_ix)
    p_b = clf.predict_proba(X_b)[:, 1]
    print_both_confusion_matrices_at_05("full_pool", y_ho_f, p_a, y_b, p_b)

    print_three_thresholds("full_pool split_a (cohort holdout)", y_ho_f, p_a, t_val, t_hold)
    print_three_thresholds(
        f"full_pool split_b (n={len(y_b)} pos={int(y_b.sum())})", y_b, p_b, t_val, t_hold
    )

    row = {
        "step_kind": "full_pool",
        "ratio_requested": -1,
        "n_neg_target": int(n_neg_full),
        "n_fp_in_constructed": int(n_fp_f),
        "n_tn_in_constructed": int(len(tn_used_f)),
        "n_fit_train": int(X_tr.shape[0]),
        "n_fit_val": int(X_va.shape[0]),
        "t_val_best_f05": float(t_val),
        "val_f05_at_t_val": float(v_f05_va),
        "t_hold_oracle_best_f05": float(t_hold),
        "hold_f05_at_t_hold": float(v_f05_hold),
        "split_a_n": int(len(y_ho_f)),
        "split_a_pos": int(y_ho_f.sum()),
        "split_b_n": int(len(y_b)),
        "split_b_pos": int(y_b.sum()),
    }
    for prefix, yv, pv in (("a", y_ho_f, p_a), ("b", y_b, p_b)):
        append_eval_row(row, prefix, yv, pv, 0.5, "t05")
        append_eval_row(row, prefix, yv, pv, t_val, "tval")
        append_eval_row(row, prefix, yv, pv, t_hold, "thold")
    sweep_rows.append(row)
else:
    print("Full-pool step skipped (identical to last ratio iteration).")

sweep_df = pd.DataFrame(sweep_rows)
_sweep_path = os.path.join(OUTPUT_DIR, "fp_followup_ratio_sweep.csv")
sweep_df.to_csv(_sweep_path, index=False)
print("\nTabPFN backend: tabpfn-client (cloud API)")
print("Saved:", _sweep_path, "rows:", len(sweep_df))



00:04 Fitting... Done!
00:03 Predicting... Done!
00:03 Predicting... Done!

ratio=1 | cohort holdout n=37 pos=18 | t_val=0.4200 (val F0.5=0.9474) | t_hold_oracle=0.7000 (hold F0.5=0.9756)

=== ratio=1 split_a (cohort holdout) ===
  t=0.5                                      t=0.500 | ROC-AUC=0.9912 PR-AUC=0.9914 F0.5=0.9184 acc=0.9459
Confusion matrix (rows=true, cols=pred):
                pred:0 (neg)   pred:1 (pos)
    true:0 (neg)           17            2   TN, FP
    true:1 (pos)            0           18   FN, TP
  t=val_best_F0.5                            t=0.420 | ROC-AUC=0.9912 PR-AUC=0.9914 F0.5=0.8824 acc=0.9189
Confusion matrix (rows=true, cols=pred):
                pred:0 (neg)   pred:1 (pos)
    true:0 (neg)           16            3   TN, FP
    true:1 (pos)            0           18   FN, TP
  t=holdout_oracle_F0.5 (optimistic)         t=0.700 | ROC-AUC=0.9912 PR-AUC=0.9914 F0.5=0.9756 acc=0.9459
Confusion matrix (rows=true, cols=pred):
                pred:0 (neg)  

## 4. Output


In [6]:
_p = os.path.join(OUTPUT_DIR, "fp_followup_ratio_sweep.csv")
print("Primary sweep table:", _p)
if len(sweep_df):
    try:
        from IPython.display import HTML, display

        _html = sweep_df.to_html(
            classes="followup_tbl", escape=False, float_format=lambda x: f"{x:.6g}"
        )
        display(
            HTML(
                "<style>.followup_tbl{font-size:11px;} "
                ".followup_tbl th,.followup_tbl td{white-space:nowrap;padding:4px 8px;}</style>"
                '<div style="max-height:480px;overflow:auto;border:1px solid #ccc;padding:8px;">'
                + _html
                + "</div>"
            )
        )
    except Exception:
        print(sweep_df.to_string())



Primary sweep table: /kaggle/working/vlst_fp_followup_output/fp_followup_ratio_sweep.csv


,step_kind,ratio_requested,n_neg_target,n_fp_in_constructed,n_tn_in_constructed,n_fit_train,n_fit_val,t_val_best_f05,val_f05_at_t_val,t_hold_oracle_best_f05,hold_f05_at_t_hold,split_a_n,split_a_pos,split_b_n,split_b_pos,a_t05_precision,a_t05_recall,a_t05_f1,a_t05_f05,a_t05_f2,a_t05_accuracy,a_t05_roc_auc,a_t05_pr_auc,a_t05_cm00,a_t05_cm01,a_t05_cm10,a_t05_cm11,a_tval_precision,a_tval_recall,a_tval_f1,a_tval_f05,a_tval_f2,a_tval_accuracy,a_tval_roc_auc,a_tval_pr_auc,a_tval_cm00,a_tval_cm01,a_tval_cm10,a_tval_cm11,a_thold_precision,a_thold_recall,a_thold_f1,a_thold_f05,a_thold_f2,a_thold_accuracy,a_thold_roc_auc,a_thold_pr_auc,a_thold_cm00,a_thold_cm01,a_thold_cm10,a_thold_cm11,b_t05_precision,b_t05_recall,b_t05_f1,b_t05_f05,b_t05_f2,b_t05_accuracy,b_t05_roc_auc,b_t05_pr_auc,b_t05_cm00,b_t05_cm01,b_t05_cm10,b_t05_cm11,b_tval_precision,b_tval_recall,b_tval_f1,b_tval_f05,b_tval_f2,b_tval_accuracy,b_tval_roc_auc,b_tval_pr_auc,b_tval_cm00,b_tval_cm01,b_tval_cm10,b_tval_cm11,b_thold_precision,b_thold_recall,b_thold_f1,b_thold_f05,b_thold_f2,b_thold_accuracy,b_thold_roc_auc,b_thold_pr_auc,b_thold_cm00,b_thold_cm01,b_thold_cm10,b_thold_cm11
0,ratio,1,92,44,48,110,37,0.42,0.947368,0.7,0.97561,37,18,5075,37,0.9,1,0.947368,0.918367,0.978261,0.945946,0.991228,0.991358,17,2,0,18,0.857143,1,0.923077,0.882353,0.967742,0.918919,0.991228,0.991358,16,3,0,18,1,0.888889,0.941176,0.97561,0.909091,0.945946,0.991228,0.991358,19,0,2,16,0.0457433,0.972973,0.0873786,0.0565149,0.192513,0.851823,0.963955,0.268678,4287,751,1,36,0.036,0.972973,0.0694311,0.0445876,0.156794,0.809852,0.963955,0.268678,4074,964,1,36,0.0730594,0.864865,0.134737,0.0894354,0.273038,0.919015,0.963955,0.268678,4632,406,5,32
1,ratio,5,460,44,416,331,110,0.85,0.784314,0.87,0.714286,111,19,4854,37,0.625,0.789474,0.697674,0.652174,0.75,0.882883,0.912471,0.77173,83,9,4,15,0.769231,0.526316,0.625,0.704225,0.561798,0.891892,0.912471,0.77173,89,3,9,10,0.818182,0.473684,0.6,0.714286,0.517241,0.891892,0.912471,0.77173,90,2,10,9,0.0493066,0.864865,0.0932945,0.0607672,0.200753,0.871858,0.935628,0.12494,4200,617,5,32,0.124402,0.702703,0.211382,0.148912,0.364146,0.960033,0.935628,0.12494,4634,183,11,26,0.128342,0.648649,0.214286,0.152866,0.358209,0.963741,0.935628,0.12494,4654,163,13,24
2,ratio,10,920,44,876,607,202,0.92,0.862069,0.93,0.744681,203,19,4578,37,0.548387,0.894737,0.68,0.594406,0.794393,0.921182,0.97254,0.807025,170,14,2,17,0.888889,0.421053,0.571429,0.727273,0.470588,0.940887,0.97254,0.807025,183,1,11,8,1,0.368421,0.538462,0.744681,0.421687,0.940887,0.97254,0.807025,184,0,12,7,0.0639098,0.918919,0.119508,0.0785219,0.25,0.890564,0.971301,0.407127,4043,498,3,34,0.327273,0.486486,0.391304,0.350195,0.44335,0.987768,0.971301,0.407127,4504,37,19,18,0.394737,0.405405,0.4,0.396825,0.403226,0.99017,0.971301,0.407127,4518,23,22,15
3,ratio,15,1380,44,1336,883,294,0.87,0.777778,0.88,0.8,295,19,4302,37,0.395349,0.894737,0.548387,0.445026,0.714286,0.905085,0.955568,0.797204,250,26,2,17,0.75,0.631579,0.685714,0.722892,0.652174,0.962712,0.955568,0.797204,272,4,7,12,0.857143,0.631579,0.727273,0.8,0.666667,0.969492,0.955568,0.797204,274,2,7,12,0.0693878,0.918919,0.129032,0.0851277,0.266458,0.893305,0.957688,0.361752,3809,456,3,34,0.252427,0.702703,0.371429,0.289532,0.517928,0.979544,0.957688,0.361752,4188,77,11,26,0.285714,0.702703,0.40625,0.32419,0.543933,0.982334,0.957688,0.361752,4200,65,11,26
4,ratio,20,1840,44,1796,1159,386,0.94,0.714286,0.9,0.8,387,19,4026,37,0.305085,0.947368,0.461538,0.352941,0.666667,0.891473,0.980692,0.800773,327,41,1,18,0.888889,0.421053,0.571429,0.727273,0.470588,0.968992,0.980692,0.800773,367,1,11,8,0.857143,0.631579,0.727273,0.8,0.666667,0.976744,0.980692,0.800773,366,2,7,12,0.0762527,0.945946,0.141129,0.093433,0.288303,0.894188,0.975866,0.438175,3565,424,2,35,0.461538,0.486486,0.473684,0.466321,0.481283,0.990065,0.975866,0.438175,3968,21,19,18,0.311688,0.648649,0.421053,0.347826,0.533333,0.983607,0.975866,0.438175,3936,53,13,24
5,ratio,25,2300,44,2256,1435,478,0.97,0.65

## 5. Saved artifact paths


In [7]:
print(
    "Artifacts written to:",
    OUTPUT_DIR,
    "\n",
    " - fp_followup_ratio_sweep.csv",
)


Artifacts written to: /kaggle/working/vlst_fp_followup_output 
  - fp_followup_ratio_sweep.csv
